# 01. Initial triangle net

This notebook plots the boreholes in the fitted logical coordinate system with marginal kernel density estimates (KDEs), alongside the initial triangle net in local XY coordinates.

**Input:** `demo_model.npz`, generated by `run_model.py`. The input directory is selected by `HG3D_MODEL_DIR` (default: `outputs/model/`).

**Output:** `original_triangle_net.png`. Figures are displayed and saved to `HG3D_FIGURE_DIR` (default: `outputs/figures/`).

**Interpretation:** The logical coordinates are centred projections used to fit the survey grid. The map panel uses the translated local XY coordinates.


## 1. Imports and configuration


In [ ]:
import os
import sys
from math import cos, sin
from pathlib import Path

# Project paths.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "run_model.py").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start this notebook from the HarmonicGeo3D directory or a subdirectory."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from scripts.generate_tri_net import (
    find_kde_peaks,
    find_top_two_kde_peaks,
    hough_transform,
    transform_along_basis,
)

# Input data.
MODEL_DATA_DIR = Path(
    os.environ.get("HG3D_MODEL_DIR", PROJECT_ROOT / "outputs" / "model")
)

CM_TO_INCH = 1 / 2.54
FONT_FAMILY = "Arial"
FIGURE_WIDTH_CM = 12.5
FIGURE_HEIGHT_CM = 8.0
U_COLOR = "#d87852"
V_COLOR = "#2dada9"
POINT_COLOR = "#2c6da4"
PLOT_RC = {}

# Figure output.
FIGURE_DIR = Path(
    os.environ.get("HG3D_FIGURE_DIR", PROJECT_ROOT / "outputs" / "figures")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data loading and logical-coordinate preparation


In [ ]:
with np.load(MODEL_DATA_DIR / "demo_model.npz", allow_pickle=False) as model_data:
    initial_nodes_xy = np.asarray(model_data["initial_nodes_xy"], dtype=float)
    initial_triangles = np.asarray(model_data["initial_triangles"], dtype=int)

# The fitted directions reproduce the logical coordinates of the initial mesh.
# Dashed lines show candidate grid axes identified by KDE peaks.

theta_peaks = find_kde_peaks(hough_transform(initial_nodes_xy)[:, 0])
theta_1, theta_2 = find_top_two_kde_peaks(theta_peaks)[:, 0]
basis_1 = np.array([cos(theta_1), sin(theta_1)])
basis_2 = np.array([cos(theta_2), sin(theta_2)])
logical_xy = transform_along_basis(
    initial_nodes_xy - initial_nodes_xy.mean(axis=0),
    basis_1,
    basis_2,
)
grid_u = find_kde_peaks(logical_xy[:, 0], bw=0.07)[:, 0]
grid_v = find_kde_peaks(logical_xy[:, 1], bw=0.07)[:, 0]

## 3. Plot the initial mesh and marginal KDEs


In [ ]:
with plt.rc_context(PLOT_RC):
    fig = plt.figure(
        figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, FIGURE_HEIGHT_CM * CM_TO_INCH), dpi=600
    )
    layout = fig.add_gridspec(
        2,
        3,
        width_ratios=(4.2, 1.0, 4.2),
        height_ratios=(0.45, 4.2),
        hspace=0.04,
        wspace=0.08,
    )
    ax_kde_u = fig.add_subplot(layout[0, 0])
    ax_grid = fig.add_subplot(layout[1, 0], sharex=ax_kde_u)
    ax_kde_v = fig.add_subplot(layout[1, 1], sharey=ax_grid)
    ax_original = fig.add_subplot(layout[1, 2])

    # The logical map and marginal KDEs share the same coordinate limits.
    u_values, v_values = logical_xy[:, 0], logical_xy[:, 1]
    u_padding = 0.07 * np.ptp(u_values) if np.ptp(u_values) > 0 else 1.0
    v_padding = 0.07 * np.ptp(v_values) if np.ptp(v_values) > 0 else 1.0
    u_limits = (u_values.min() - u_padding, u_values.max() + u_padding)
    v_limits = (v_values.min() - v_padding, v_values.max() + v_padding)

    for values, limits, kde_ax, horizontal, color in (
        (u_values, u_limits, ax_kde_u, True, U_COLOR),
        (v_values, v_limits, ax_kde_v, False, V_COLOR),
    ):
        support = np.linspace(*limits, 300)
        density = stats.gaussian_kde(values, bw_method=0.07)(support)
        if horizontal:
            kde_ax.fill_between(support, density, color=color, alpha=0.42)
            kde_ax.plot(support, density, color=color, linewidth=0.8)
        else:
            kde_ax.fill_betweenx(support, density, color=color, alpha=0.42)
            kde_ax.plot(density, support, color=color, linewidth=0.8)

    for value in grid_u:
        ax_grid.axvline(
            value, color=U_COLOR, linestyle="--", linewidth=0.7, alpha=0.58, zorder=1
        )
    for value in grid_v:
        ax_grid.axhline(
            value, color=V_COLOR, linestyle="--", linewidth=0.7, alpha=0.58, zorder=1
        )
    ax_grid.scatter(
        logical_xy[:, 0],
        logical_xy[:, 1],
        s=10,
        color=POINT_COLOR,
        edgecolor="white",
        linewidth=0.5,
        zorder=2,
    )

    ax_grid.set_xlim(*u_limits)
    ax_grid.set_ylim(*v_limits)
    ax_grid.set_aspect("equal", adjustable="box")
    ax_grid.text(
        0.02,
        0.98,
        "(a)",
        transform=ax_grid.transAxes,
        va="top",
        ha="left",
        family=FONT_FAMILY,
        fontsize=7,
        clip_on=True,
    )
    for spine in ax_grid.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("#333333")

    ax_kde_u.set_xlim(*u_limits)
    ax_kde_u.tick_params(axis="x", labelbottom=False)
    ax_kde_v.set_ylim(*v_limits)
    ax_kde_v.tick_params(axis="y", labelleft=False)

    for spine in ax_kde_u.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("#333333")
    for spine in ax_kde_v.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("#333333")

    # Panel (b) uses the initial mesh in local XY coordinates.
    x_padding = np.ptp(initial_nodes_xy[:, 0]) * 0.03
    y_padding = np.ptp(initial_nodes_xy[:, 1]) * 0.03
    x_limits = (
        initial_nodes_xy[:, 0].min() - x_padding,
        initial_nodes_xy[:, 0].max() + x_padding,
    )
    y_limits = (
        initial_nodes_xy[:, 1].min() - y_padding,
        initial_nodes_xy[:, 1].max() + y_padding,
    )
    ax_original.triplot(
        initial_nodes_xy[:, 0],
        initial_nodes_xy[:, 1],
        initial_triangles,
        color="0.38",
        linewidth=0.40,
        zorder=1,
    )
    ax_original.scatter(
        initial_nodes_xy[:, 0],
        initial_nodes_xy[:, 1],
        s=10,
        facecolor=POINT_COLOR,
        edgecolor="white",
        linewidth=0.5,
        zorder=2,
    )
    ax_original.set_xlim(*x_limits)
    ax_original.set_ylim(*y_limits)
    ax_original.set_aspect("equal", adjustable="box")
    ax_original.yaxis.set_label_position("right")
    ax_original.yaxis.tick_right()
    ax_original.text(
        0.02,
        0.98,
        "(b)",
        transform=ax_original.transAxes,
        va="top",
        ha="left",
        family=FONT_FAMILY,
        fontsize=7,
        clip_on=True,
    )

    for spine in ax_original.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("#333333")

    for ax in (ax_grid, ax_original):
        ax.set_axisbelow(True)
        ax.grid(axis="both", alpha=0.50, color="lightgray", linewidth=0.25)
        ax.tick_params(
            axis="both",
            which="both",
            top=True,
            right=True,
            direction="in",
            labelsize=4,
            length=2,
            width=0.5,
        )
        ax.ticklabel_format(style="plain", useOffset=False)
    for ax in (ax_kde_u, ax_kde_v):
        ax.tick_params(
            axis="both",
            which="both",
            top=True,
            right=True,
            direction="in",
            labelsize=3.5,
            length=1.2,
            width=0.3,
        )
        ax.ticklabel_format(style="plain", useOffset=False)

    # Equal aspect narrows the logical map within its GridSpec cell.
    # The top KDE width follows the rendered map width.
    fig.canvas.draw()
    grid_bbox = ax_grid.get_position()
    kde_bbox = ax_kde_u.get_position()
    ax_kde_u.set_position([grid_bbox.x0, kde_bbox.y0, grid_bbox.width, kde_bbox.height])
    # The middle KDE spans the map height. Its density-axis length matches
    # the top KDE's density-axis height in physical units.
    side_bbox = ax_kde_v.get_position()
    side_gap = 0.006
    density_axis_width = kde_bbox.height * fig.get_figheight() / fig.get_figwidth()
    ax_kde_v.set_position(
        [grid_bbox.x1 + side_gap, side_bbox.y0, density_axis_width, side_bbox.height]
    )
    # A smaller gap brings the local XY map closer to the marginal KDE.
    side_bbox = ax_kde_v.get_position()
    planar_bbox = ax_original.get_position()
    planar_gap = 0.008
    ax_original.set_position(
        [
            side_bbox.x1 + planar_gap,
            planar_bbox.y0,
            planar_bbox.width,
            planar_bbox.height,
        ]
    )
    fig.canvas.draw()

    plt.savefig(FIGURE_DIR / "original_triangle_net.png", dpi=600)
    plt.show()